<a href="https://colab.research.google.com/github/azcsprof/ASU-CSE475-SS25/blob/Unit-3-ICE-1/Unit_3_ICE_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown, Checkbox
from IPython.display import display, clear_output

# Reproducibility
torch.manual_seed(42)

In [2]:
# A single-layer perceptron: 1 input → 1 output
class SingleLayerPerceptron(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(1, 1)  # Linear function: Y = WX + b

    def forward(self, x):
        return self.fc(x)

In [3]:
def train_model(model, optimizer, X_data, Y_data, epochs=10, label="Model"):
    loss_fn = nn.MSELoss()
    weight_hist, bias_hist, loss_hist = [], [], []

    print(f"\n Training {label}:")

    for epoch in range(epochs):
        total_loss = 0.0

        # Loop over all training pairs (X_i, Y_i)
        for x, y in zip(X_data, Y_data):
            x = x.unsqueeze(0)  # Reshape from scalar to 1D tensor
            y = y.unsqueeze(0)

            optimizer.zero_grad()           # Reset gradient
            pred = model(x)                 # Forward pass
            loss = loss_fn(pred, y)         # Compare output to true value
            loss.backward()                 # Backward pass
            optimizer.step()                # Update weights

            total_loss += loss.item()

        # Track learning
        w = model.fc.weight.item()
        b = model.fc.bias.item()
        weight_hist.append(w)
        bias_hist.append(b)
        loss_hist.append(total_loss)

        print(f"Epoch {epoch+1:02d} | Loss: {total_loss:.4f} | w: {w:.3f} | b: {b:.3f}")

    return weight_hist, bias_hist, loss_hist

In [4]:
def plot_loss(loss_hist, title="Loss over Epochs"):
    plt.figure(figsize=(6, 4))
    plt.plot(loss_hist, marker='o', label="Epoch Loss")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Total Loss")
    plt.grid(True)
    plt.legend()
    plt.show()

In [5]:
# Simulate 100 values from X ∈ [-1, 1]
X = 2 * torch.rand(100) - 1
Y_clean = -2 * X + 5  # Linear function with slope -2, intercept 5

In [12]:
# -- Interactive training function --
def run_interactive_training(learning_rate, optimizer_type, add_noise, epochs):
    clear_output(wait=True)

    Y = Y_clean + torch.normal(0, 0.1, size=X.size()) if add_noise else Y_clean
    model = SingleLayerPerceptron()

    if optimizer_type == "SGD":
        optimizer = optim.SGD(model.parameters(), lr=learning_rate)
    elif optimizer_type == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    _, _, loss_hist = train_model(
        model, optimizer, X, Y,
        epochs=epochs,
        label=f"{optimizer_type} | LR={learning_rate:.3f} | Epochs={epochs} | Noise={add_noise}"
    )

    plot_loss(loss_hist, title=f"Loss | {optimizer_type}, LR={learning_rate:.3f}, Epochs={epochs}, Noise={add_noise}")

In [15]:
from ipywidgets import IntSlider

# Interactive widget definitions
learning_rate_slider = FloatSlider(value=0.01, min=0.001, max=0.2, step=0.005, description='Learning Rate')
optimizer_dropdown = Dropdown(options=["SGD", "Adam"], value="SGD", description="Optimizer")
noise_checkbox = Checkbox(value=False, description="Add Noise")
epoch_slider = IntSlider(value=10, min=1, max=100, step=1, description='Epochs')  # IntSlider for integer epochs

# Launch interactive interface
interact(
    run_interactive_training,
    learning_rate=learning_rate_slider,
    optimizer_type=optimizer_dropdown,
    add_noise=noise_checkbox,
    epochs=epoch_slider
)

interactive(children=(FloatSlider(value=0.01, description='Learning Rate', max=0.2, min=0.001, step=0.005), Dr…

<function __main__.run_interactive_training(learning_rate, optimizer_type, add_noise, epochs)>

### **Interpretation Guide – Training With Noise**

---

#### What You’re Seeing:
- **Initial Epoch (0)**: Loss is very high (~140), which is expected — the model starts with random weights and has no knowledge of the function.
- **Epochs 1–2**: Loss drops dramatically. The model rapidly fits the overall trend in the data (Y ≈ -2X + 5).
- **Epochs 3–20**: Loss stabilizes around **1.32**, rather than approaching zero.

---

#### Why This Happens:

| Observation                  | Explanation                                                                 |
|------------------------------|------------------------------------------------------------------------------|
| Sharp drop in early epochs   | The model quickly captures the main structure of the data.                  |
| Non-zero loss plateau        | Random noise in the target values prevents perfect predictions.             |
| Stable weights after epoch 2 | Indicates convergence to the best-fit line under noisy conditions.          |

---

#### Reflection Questions:
- How does the model's behavior change if **noise is disabled**?
- What does a **non-zero but stable loss** tell us about the model's learning?
- What happens if you increase **epochs beyond 20** with noise still on?
- If weights stop changing, but loss doesn't reach 0 — what does that indicate?

---

#### Key Insight:
> **Noise limits model performance.** Even when the model finds the correct underlying pattern, it cannot eliminate randomness. The goal is not perfection — it's to find the **signal within the noise**.